- Object: Classification of point clouds (vase or monitor)
- Dataset:
  - ModelNet40
  - Only used vase and monitor subset
- Model : MLP
- Description:
  - Preprocessing:
    - ModelNet40's dataset is based on .off format ; Convert .off to point cloud
    - Normalization
  - Model
    - Training
    - Testing
  - Model Evaluation
    - accuracy

In [1]:
!pip install open3d torch torchvision numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 52.8 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


In [19]:
import os
import trimesh
import open3d as o3d
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


# Dataset

In [4]:
!wget http://modelnet.cs.princeton.edu/ModelNet40.zip
!unzip ModelNet40.zip -d /content/ # the dataset stored temporarily

Streaming output truncated to the last 5000 lines.
  inflating: /content/ModelNet40/monitor/train/monitor_0120.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0199.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0285.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0227.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0281.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0067.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0368.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0090.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0436.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0137.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0011.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0288.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0284.off  
  inflating: /content/ModelNet40/monitor/train/monitor_0325.off  
  inflating: /content/Mod

In [9]:
!pip install trimesh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.3/740.3 kB 31.0 MB/s eta 0:00:00


In [13]:
import trimesh

mesh_tm = trimesh.load("/content/ModelNet40/monitor/train/monitor_0120.off")

mesh = o3d.geometry.TriangleMesh(
    o3d.utility.Vector3dVector(mesh_tm.vertices),
    o3d.utility.Vector3iVector(mesh_tm.faces)
)

mesh.compute_vertex_normals()
o3d.visualization.draw_plotly([mesh])

In [15]:
# mesh에서 포인트 클라우드 생성
pcd = mesh.sample_points_uniformly(number_of_points=2048)
print(pcd)

PointCloud with 2048 points.


In [16]:
o3d.visualization.draw_plotly([pcd])

# Preprocessing

In [17]:
def load_off_as_pointcloud(path, n_points=1024):
  mesh_tm = trimesh.load(path)

  mesh = o3d.geometry.TriangleMesh(
      o3d.utility.Vector3dVector(mesh_tm.vertices),
      o3d.utility.Vector3iVector(mesh_tm.faces)
  )

  pcd = mesh.sample_points_uniformly(number_of_points=n_points)
  points = np.asarray(pcd.points)

  # Normalization
  points = points - points.mean(axis=0) # ignore the location of the object
  scale = np.max(np.linalg.norm(points, axis=1))
  points = points / scale # fit the object in a unit sphere

  return points.astype(np.float32)

In [24]:
class ModelNetBinaryDataset(Dataset):
  def __init__(self, root_dir, classes):
    self.samples = []
    self.labels = []

    for label, cls in enumerate(classes):
      cls_dir = os.path.join(root_dir, cls, "train")
      for fname in os.listdir(cls_dir)[:50]:  # decrease the size of dataset
        path = os.path.join(cls_dir, fname)
        self.samples.append(load_off_as_pointcloud(path))
        self.labels.append(label)

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx):
    return torch.tensor(self.samples[idx]), torch.tensor(self.labels[idx])

In [25]:
dataset = ModelNetBinaryDataset(
  root_dir="/content/ModelNet40",
  classes=["monitor", "vase"]
)

loader = DataLoader(dataset, batch_size=8, shuffle=True)

# Model

## Training

In [20]:
import torch.nn as nn
import torch.nn.functional as F

class SimplePointNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.mlp1 = nn.Linear(3, 64)
        self.mlp2 = nn.Linear(64, 128)
        self.mlp3 = nn.Linear(128, 256)

        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (B, N, 3)
        x = F.relu(self.mlp1(x))
        x = F.relu(self.mlp2(x))
        x = F.relu(self.mlp3(x))

        x = torch.max(x, dim=1)[0]  # global max pooling
        x = F.relu(self.fc1(x))
        return self.fc2(x)


In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimplePointNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [27]:
for epoch in range(5):
    total_loss = 0
    for points, labels in loader:
        points, labels = points.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(points)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


Epoch 1, Loss: 9.2013
Epoch 2, Loss: 8.9597
Epoch 3, Loss: 8.7388
Epoch 4, Loss: 8.1913
Epoch 5, Loss: 6.4523


## Testing

In [28]:
model.eval()
points, label = dataset[0]
points = points.unsqueeze(0).to(device)

with torch.no_grad():
    pred = model(points)
    print("Predicted:", pred.argmax(dim=1).item(),
          "GT:", label.item())

Predicted: 0 GT: 0


# Evaluation

In [29]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for points, labels in loader:
        points, labels = points.to(device), labels.to(device)
        outputs = model(points)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Accuracy: {accuracy*100:.2f}%")


Accuracy: 94.00%
